In [1]:
!pip install torch

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/123.0 MB ? eta -:--:--
   - -------------------------------------- 3.7/123.0 MB 13.0 MB/s eta 0:00:10
   --- ------------------------------------ 10.5/123.0 MB 21.2 MB/s eta 0:00:06
   ----- ---------------------------------- 17.0/123.0 MB 24.1 MB/s eta 0:00:05
   ------ --------------------------------- 19.7/123.0 MB 24.7 MB/s eta 0:00:05
   -------- ------------------------------- 27.0/123.0 MB 23.8 MB/s eta 0:00:05
   ---------- ----------------------------- 31.5/123.0 

In [2]:
import torch
import torch.nn.functional as F

c:\Users\miras\OneDrive\Документы\GitHub\GPT-from-Scratch-PyTorch-\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [7]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [8]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [9]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: (''.join([itos[i] for i in l]))

print(encode('Hello, World!'))
print(decode([20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]))

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [10]:
print(len(text))

1115394


In [11]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [12]:
n = int(len(text)*0.9)
train_data = data[:n]
val_data = data[n:]

In [13]:
block_size = 8
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t:t+1]
  print(context, 'the next int', target)

tensor([18]) the next int tensor([47])
tensor([18, 47]) the next int tensor([56])
tensor([18, 47, 56]) the next int tensor([57])
tensor([18, 47, 56, 57]) the next int tensor([58])
tensor([18, 47, 56, 57, 58]) the next int tensor([1])
tensor([18, 47, 56, 57, 58,  1]) the next int tensor([15])
tensor([18, 47, 56, 57, 58,  1, 15]) the next int tensor([47])
tensor([18, 47, 56, 57, 58,  1, 15, 47]) the next int tensor([58])


In [15]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(0, len(data) - block_size -1, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print(xb.shape)
xb, yb

torch.Size([4, 8])


(tensor([[53, 59,  6,  1, 58, 56, 47, 40],
         [49, 43, 43, 54,  1, 47, 58,  1],
         [13, 52, 45, 43, 50, 53,  8,  0],
         [ 1, 39,  1, 46, 53, 59, 57, 43]]),
 tensor([[59,  6,  1, 58, 56, 47, 40, 59],
         [43, 43, 54,  1, 47, 58,  1, 58],
         [52, 45, 43, 50, 53,  8,  0, 26],
         [39,  1, 46, 53, 59, 57, 43,  0]]))

In [16]:
import torch.nn as nn
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    if targets is None:
      loss = None
    
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :] # taking the last time step
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1) # (Batch_size, 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx




m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(loss)

print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

tensor(4.8948, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [17]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [30]:
batch_size = 32
for _ in range(100):
  xb, yb = get_batch('train')
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

2.73960280418396


In [35]:
print(decode( m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist() ))


Awnnd! aySond qHHAUZ3ncametr w buory.
AnUt,
ane w, boreawf!KIABRWh hing thoranord:
IOMII sgeteeZRI'dastoulabe trkinQbeatXY:litrereeaKIO'do, prszereralFo!

navvit mR:J.
LT:CowiYh prs momJwanf?sceapenD.
WhageGBas t is t oathin lpiliAUFate nggowilll; ud,
W:
ObrE3gicans tulnisonve inctaleir thR
Ti.


m egulkINzCO fry f thtonkto!

Whse m sp-II:CORCKIXofe y TDWIZAcieiro ibasiz aXeh bamal'Th myd ayFacretl'llpois! flvr;athisexO Hoot t y lllt ou E:CExld,Sin ims&sBRC&llvesem.
Winoriy ID.
xWiswL&up.
Bye be
